# Regime Detection Notebook

Regime-conditional VaR/CVaR pipeline for energy markets (WTI, Brent, Henry Hub) using macro indicators.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from regime_detection.ingestion import get_merged_df
from regime_detection.features import feature_pipeline
from regime_detection.stationarity import stationarity_pipeline
from regime_detection.regression import (
    fit_commodity_models, fit_rolling_commodity_models,
    ljung_box, durbin_watson_stat, significant_coefficients,
)
from regime_detection.changepoints import (
    run_crops, select_plateau_changepoints, label_regimes, regime_summary,
)
from regime_detection.risk import (
    add_rolling_var_columns, regime_conditional_risk,
    unconditional_risk, compute_breach_rates,
)
from regime_detection.backtest import (
    kupiec_test, kupiec_regime_conditional, kupiec_unconditional,
)

## Data Ingestion & Feature Engineering

In [ ]:
df_raw = get_merged_df()
df_features = feature_pipeline(df_raw)

split_ratio = 0.8
split_index = int(len(df_features) * split_ratio)
df_train, df_test = df_features.iloc[:split_index], df_features.iloc[split_index:]

print(f"Train: {len(df_train)} rows, Test: {len(df_test)} rows")

## Stationarity Transforms

In [ ]:
df_train = stationarity_pipeline(df_train)
df_model = df_train.dropna()
print(f"Modelling sample: {len(df_model)} rows")

## OLS Regression

In [ ]:
feature_cols = [
    "WCESTUS1_1_w_change_stationary",  # inventory surprise
    "RBRTE_RWTC_diff",                  # brent-wti spread
    "VIXCLS",                           # risk sentiment
    "T10Y2Y_stationary",                # yield curve
    "DTWEXBGS_stationary",              # dollar index
]

models = fit_commodity_models(df_model, feature_cols)

for name, res in models.items():
    print(f"--- {name.upper()} ---")
    print(f"R²: {res['ols'].rsquared:.4f}")
    print(f"Residuals mean: {res['residuals'].mean():.6f}, std: {res['residuals'].std():.4f}")

### Evaluating Autocorrelation with the Ljung-Box Test

The Ljung-Box test's null hypothesis is that residuals are not autocorrelated within a window of the last n observations. Rejecting it confirms the need for HAC standard errors.

In [ ]:
for name, res in models.items():
    lb = ljung_box(res["residuals"])
    print(f"{name.upper()} Ljung-Box:")
    print(lb)

## Regression T-Statistics with HAC Standard Errors

Newey-West (HAC) estimates the covariance matrix for OLS when its assumptions of constant variance and no autocorrelation are violated. If residuals exhibit positive autocorrelation, regular OLS underestimates standard errors, inflating t-statistics.

In [ ]:
for name, res in models.items():
    ols_sig = significant_coefficients(res["ols"])
    hac_sig = significant_coefficients(res["hac"])
    print(f"{name.upper()}:")
    print(f"  OLS significant: {ols_sig}")
    print(f"  HAC significant: {hac_sig}")
    print(f"  T-stat delta:{res['ols'].tvalues - res['hac'].tvalues}")

## Durbin-Watson Statistic

The Durbin-Watson statistic ranges from 0 to 4. A value of 2 indicates no autocorrelation, below 2 indicates positive autocorrelation, and above 2 indicates negative autocorrelation.

In [ ]:
for name, res in models.items():
    dw = durbin_watson_stat(res["residuals"])
    print(f"{name.upper()} Durbin-Watson: {dw:.3f}")

## ACF / PACF Plots

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
for i, (name, res) in enumerate(models.items()):
    plot_acf(res["residuals"], ax=axes[i][0], title=f"{name.upper()} ACF", lags=20)
    plot_pacf(res["residuals"], ax=axes[i][1], title=f"{name.upper()} PACF", lags=20)
plt.tight_layout()
plt.show()

## Residual Plots

In [ ]:
plt.figure(figsize=(8, 8))
for name, color in [("henry", "b"), ("brent", "r"), ("wti", "g")]:
    plt.plot(models[name]["residuals"], color, label=f"{name.upper()} Residuals")
plt.xlabel("Date")
plt.ylabel("Residual (Log Basis)")
plt.legend(loc="upper left")
plt.show()

## Rolling OLS

We fit Rolling OLS across 52, 78, and 104-week windows to observe time-varying coefficient paths. Of particular interest is B1 (inventory coefficient), which we later use for changepoint detection.

In [ ]:
windows = [52, 78, 104]
rolling_results = fit_rolling_commodity_models(df_model, feature_cols, windows=windows)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 24), sharey=True)
for num, window in enumerate(windows):
    ax = axes[num]
    for series, color in [("wti", "g"), ("brent", "b"), ("henry", "r")]:
        params = rolling_results[series][window].params
        ax.plot(params.iloc[window:, 1], color, label=series.upper())
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.set_xlabel("Date")
    ax.set_ylabel("B1")
    ax.set_title(f"B1 over Time ({window}-week windows)")
    ax.legend(loc="upper left")
fig.autofmt_xdate()
plt.show()

### PELT Changepoint Detection

We identify regimes using changepoint detection on the 12-week rolling volatility of WTI returns and the rolling B1 (inventory) coefficient from the 78-week Rolling OLS. CROPS binary search discovers all distinct PELT segmentations across the penalty range [ln(n), 100·ln(n)].

In [ ]:
window = 78
pelt_series = [
    df_model["RWTC_log_return_12_returns_rol_vol"].values,
    rolling_results["wti"][window].params.iloc[window:, 1].values,
]

crops_results = {}
for i, signal in enumerate(pelt_series):
    crops_results[i] = run_crops(signal)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
titles = ["WTI 12-Week Rolling Volatility", "B1 (Inventory) Coefficient"]

for idx in range(2):
    penalties = sorted(crops_results[idx].keys())
    n_cpts = [len(crops_results[idx][p]) - 1 for p in penalties]
    axes[idx].plot(penalties, n_cpts, "o-", markersize=4)
    axes[idx].set_xlabel("Penalty")
    axes[idx].set_ylabel("Number of Changepoints")
    axes[idx].set_title(f"{titles[idx]}: Changepoints vs Penalty")
    axes[idx].set_xscale("log")
plt.tight_layout()
plt.show()

The CROPS plot reveals the penalty ranges where the segmentation is stable. We select the widest plateau for each series to determine our changepoints.

## Regime Labeling

Using the changepoints identified from the CROPS analysis on WTI 12-week rolling volatility, we segment  into contiguous regimes.

In [ ]:
vol_changepoints = select_plateau_changepoints(crops_results[0])
print(f"Selected changepoints: {vol_changepoints}")

df_model = label_regimes(df_model, vol_changepoints)
regime_summary(df_model)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.Set2(np.linspace(0, 1, df_model["regime"].nunique()))

for regime_id in sorted(df_model["regime"].unique()):
    mask = df_model["regime"] == regime_id
    ax.fill_between(
        df_model.index, 0,
        df_model["RWTC_log_return_12_returns_rol_vol"].max() * 1.05,
        where=mask, alpha=0.2, color=colors[regime_id], label=f"Regime {regime_id}",
    )
ax.plot(df_model.index, df_model["RWTC_log_return_12_returns_rol_vol"], "k-", linewidth=0.8)
ax.set_title("WTI 12-Week Rolling Volatility with Regime Labels")
ax.set_xlabel("Date")
ax.set_ylabel("Rolling Volatility")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## VaR Calculation

We compute Value at Risk (VaR) at the 5th percentile using a rolling historical simulation approach across 52, 78, and 104-week windows.

In [ ]:
df_model = add_rolling_var_columns(df_model, windows=[52, 78, 104])

In [ ]:
# VaR distribution by window
var_dict = {
    w: df_model.loc[:, df_model.columns.str.contains(f"{w}_week_var")]
    for w in [52, 78, 104]
}
fig, axes = plt.subplots(3, 1, figsize=(8, 12))
for idx, (weeks, var) in enumerate(var_dict.items()):
    sns.kdeplot(data=var, fill=True, alpha=0.4, ax=axes[idx])
    axes[idx].set_title(f"{weeks}-Week Value-at-Risk Distribution (5th percentile)")
plt.tight_layout()
plt.show()

In [ ]:
# Log returns distribution
log_returns_df = pd.DataFrame({
    "RWTC": df_model["RWTC_log_return_stationary"],
    "RBRTE": df_model["RBRTE_log_return_stationary"],
    "RNGWHHD": df_model["RNGWHHD_log_return_stationary"],
})
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
sns.kdeplot(data=log_returns_df, fill=True, alpha=0.4, ax=ax)
ax.set_title("Log Returns Distribution")
plt.show()

In [ ]:
# VaR vs Log Returns time series
series_map = {"RWTC": "RWTC", "RBRTE": "RBRTE", "RNGWHHD": "RNGWHHD"}
fig, axes = plt.subplots(3, 1, figsize=(8, 16))
for idx, (ticker, _) in enumerate(series_map.items()):
    cols = [c for c in df_model.columns if ticker in c and ("var" in c or "log_return_stationary" in c)]
    sns.lineplot(data=df_model[cols], alpha=0.4, ax=axes[idx])
    axes[idx].set_title(f"{ticker} VaR vs Log Returns")
plt.tight_layout()
plt.show()

We note that every time the log returns fall below our VaR, we have a breach. Since we set the VaR at the bottom 5th percentile, we expect a breach rate of around 5%.

In [ ]:
breach_df = compute_breach_rates(df_model, windows=[52, 78, 104])
print(breach_df)

sns.barplot(data=breach_df, x="ticker", hue="window", y="breach_rate", palette="crest")
plt.axhline(0.05, ls="--", color="red", label="Expected (5%)")
plt.legend()
plt.show()

The unconditional breach rates range from 0.1% to 1%, far below the expected 5%. The unconditional VaR is systematically overconservative because it mixes returns from different volatility regimes into a single distribution.

## Regime-Conditional VaR and CVaR

Now that we have assigned regime labels, we compute VaR and CVaR (Expected Shortfall) within each regime separately. This avoids the distributional mixing that made unconditional VaR overconservative.

In [ ]:
regime_risk_df = regime_conditional_risk(df_model)
print(regime_risk_df.to_string(index=False))

print("Unconditional VaR (full sample)")
uncond = unconditional_risk(df_model)
print(uncond.to_string(index=False))

In [ ]:
# Regime-conditional VaR bar chart
series_names = ["RWTC", "RBRTE", "RNGWHHD"]
return_cols = {s: f"{s}_log_return_stationary" for s in series_names}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for idx, series in enumerate(series_names):
    subset = regime_risk_df[regime_risk_df["series"] == series]
    x = subset["regime"].values
    axes[idx].bar(x - 0.15, subset["VaR_5pct"].values, width=0.3, label="VaR (5th pct)", color="steelblue")
    axes[idx].bar(x + 0.15, subset["CVaR"].values, width=0.3, label="CVaR", color="salmon")
    uncond_var = df_model[return_cols[series]].quantile(0.05)
    axes[idx].axhline(uncond_var, ls="--", color="black", alpha=0.6, label=f"Unconditional VaR ({uncond_var:.3f})")
    axes[idx].set_xlabel("Regime")
    axes[idx].set_title(series)
    axes[idx].set_xticks(range(df_model["regime"].nunique()))
    axes[idx].legend(fontsize=7)
axes[0].set_ylabel("Log Return Threshold")
fig.suptitle("Regime-Conditional VaR and CVaR vs Unconditional VaR", fontsize=13)
plt.tight_layout()
plt.show()

## Kupiec Backtest

The Kupiec proportion-of-failures (POF) test evaluates whether the observed breach rate is consistent with the expected rate (5%). The null hypothesis is that the true breach probability equals the target. The test statistic follows a chi-squared distribution with 1 degree of freedom.

In [ ]:
rc_results = kupiec_regime_conditional(df_model)
uc_results = kupiec_unconditional(df_model, windows=[52, 78, 104])

print("=== Regime-Conditional Kupiec Results ===")
print(rc_results[["regime", "series", "n_obs", "breach_rate", "LR_stat", "p_value", "reject_H0"]].to_string(index=False))

print("=== Unconditional Kupiec Results ===")
print(uc_results[["type", "series", "n_obs", "breach_rate", "LR_stat", "p_value", "reject_H0"]].to_string(index=False))

In [ ]:
# Combine for visualization
kupiec_df = pd.concat([rc_results, uc_results], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, series in enumerate(series_names):
    uc_data = kupiec_df[(kupiec_df["type"].str.startswith("unconditional")) & (kupiec_df["series"] == series)]
    rc_data = kupiec_df[(kupiec_df["type"] == "regime-conditional") & (kupiec_df["series"] == series)]

    uc_labels = [t.replace("unconditional_", "") for t in uc_data["type"]]
    axes[idx].bar(np.arange(len(uc_labels)) - 0.2, uc_data["breach_rate"].values,
                  width=0.4, label="Unconditional", color="steelblue", alpha=0.7)
    rc_mean = rc_data["breach_rate"].mean()
    axes[idx].bar(len(uc_labels) + 0.4, rc_mean,
                  width=0.4, label="Regime-Cond. (mean)", color="salmon", alpha=0.7)
    axes[idx].axhline(0.05, ls="--", color="red", alpha=0.8, label="Expected (5%)")
    axes[idx].set_xticks(list(range(len(uc_labels))) + [len(uc_labels)])
    axes[idx].set_xticklabels(uc_labels + ["Regime Cond."], fontsize=8)
    axes[idx].set_title(series)
    axes[idx].set_ylabel("Breach Rate")
    axes[idx].legend(fontsize=7)
fig.suptitle("Kupiec Backtest: Unconditional vs Regime-Conditional Breach Rates", fontsize=13)
plt.tight_layout()
plt.show()

### Kupiec Backtest Results

The Kupiec test confirms the failure of unconditional VaR: all unconditional rolling-window configurations reject the null hypothesis that the breach rate equals 5%, with observed rates of 0.1-1%. The unconditional VaR is systematically overconservative because it mixes returns from different volatility regimes into a single distribution.

Regime-conditional VaR, by construction, achieves breach rates closer to the 5% target within each regime, producing a risk estimate that is neither overly conservative nor overly aggressive for the current market environment.